In [ ]:
# Importing necessary libraries for ingestion layer

import pandas as pd
from snowflake.snowpark.context import get_active_session

# Establish Snowpark session and stage reference
session = get_active_session()
stage = "@CRIME_PIPELINE.RAW.CRIME_STAGE"

In [ ]:
# Using a test batch of one month to test ingestion before full implementation

months = [
    "2026-01"
]

selected_forces = [
    "west-midlands",
    "thames-valley",
    "surrey",
    "dyfed-powys"
]

# List all files currently in the stage
list_df = session.sql(f"LIST {stage}").to_pandas()
list_df.columns = [c.strip().strip('"').lower() for c in list_df.columns]

batch_frames = []

for month in months:
    # Filter stage files to this month and selected forces
    month_files = [
        row["name"].split("/")[-1]
        for _, row in list_df.iterrows()
        if month in row["name"]
        and any(force in row["name"] for force in selected_forces)
    ]

    for filename in month_files:
        # Read file directly from stage into pandas
        df = pd.read_csv(
            session.file.get_stream(f"{stage}/{filename}"),
            dtype=str,
            low_memory=False
        )

        # Metadata for validation and traceability
        df["source_month"] = month
        df["source_file"]  = filename

        batch_frames.append(df)

        print(f"Loaded: {filename}")

crime_raw = pd.concat(batch_frames, ignore_index=True)

In [ ]:
# Quick validation checks after ingestion layer complete

print("Total rows:", len(crime_raw))
print("Files loaded:", crime_raw["source_file"].nunique())
print("Months loaded:", crime_raw["source_month"].nunique())

In [ ]:
# Inspecting the data

crime_raw.head()